# 00 · Setup and sanity checks

**Katip lab, notebook 0 of 3.** About 3 minutes.

This notebook makes sure that your machine reproduces Katip's published numbers *before* we measure
anything new. A lab that cannot reproduce its own baseline cannot claim an improvement.

You will:

1. check the Python environment,
2. download the real-text test data (Turkish Wiki NER, public, CC BY-SA 4.0, ~6 MB),
3. prove that the lab's model runner gives **exactly** the same answers as the code the browser runs,
4. reproduce the README number for Katip Tiny (364 of 365 names on the blind set).

If any checkpoint (cells with `assert`) fails, stop and fix it before going on.

## 0.1 · Environment

Run Jupyter from the repository root with the BERTurk virtual environment (it already has torch,
transformers, onnx and onnxruntime). In Terminal, one line at a time:

```
cd /Users/f1k0/Development/Personal/Projects/active/katip
source ml/berturk/.venv/bin/activate
pip install -r lab/requirements.txt
jupyter lab lab/
```

In [ ]:
import os, sys, platform, json, time
import katiplab as K

print("python      ", sys.version.split()[0], platform.machine())
print("repo root   ", K.ROOT)
print("cpu cores   ", os.cpu_count())
import torch, transformers, onnxruntime, tokenizers
for m in (torch, transformers, onnxruntime, tokenizers):
    print(f"{m.__name__:12}", m.__version__)
print("BERTurk run ", "found" if (K.BERTURK / "onnx").exists() else "not found (optional)")

## 0.2 · Real-text data

Katip's own test sets are synthetic: generated from templates, with a known answer key. They are
useful, but a model can learn the templates instead of the task. To see how Katip behaves on text
it was **not** designed around, we use **Turkish Wiki NER** (Altinok, ACL 2023): 20,000 sentences
from Turkish Wikipedia, annotated by people, with `PERSON` among 19 entity types.

| split | sentences | use in this lab |
| --- | --- | --- |
| train | 18,000 | training material in notebook 2 |
| dev | 1,000 | all analysis and every choice we make |
| test | 1,000 | **sealed** until notebook 3, scored once |

The download happens once; the file hashes go into `data/wikiner/MANIFEST.json` so a reader can
check that they used the same data.

In [ ]:
manifest = K.get_wikiner()
manifest

In [ ]:
dev = K.load_wiki("dev")
ex = next(d for d in dev if sum(t == "PERSON" for *_, t in d["spans"]) >= 2)
print(ex["text"])
for a, b, t in ex["spans"]:
    print(f"  {t:8} {ex['text'][a:b]}")

## 0.3 · Checkpoint: the lab runs the same code as the browser

`katiplab.Tagger` is a small re-implementation of the browser's `ner.js` (the same as
`ml/eval_system.py`, but able to hold several models at once). Here we run both on the 80 blind
documents and require identical spans.

In [ ]:
os.environ["KATIP_MODEL"] = str(K.TINY)
sys.path.insert(0, str(K.ROOT / "ml"))
import eval_system as ES

tiny = K.Tagger(K.TINY)
blind = K.load_synth("blind")
diffs = sum(ES.ner(d["text"]) != tiny(d["text"]) for d in blind)
print("documents with different spans:", diffs)
assert diffs == 0, "the lab tagger does not match the browser port"

## 0.4 · Checkpoint: reproduce the README

The README says Katip Tiny (rules + 2.6 MB tagger) finds **364 of 365** names on the blind set and
Katip Rules finds **359**.

In [ ]:
for name, det in [("Katip Rules", K.rules()), ("Katip Tiny", K.pipeline(tiny))]:
    g, s, secs = K.score(blind, det)
    r = K.summary(g, s)
    print(f"{name:12} names {r['found']}/{r['gold']}  false names {r['false']}  ({1000 * secs / len(blind):.0f} ms/doc)")
assert r["found"] == 364 and r["gold"] == 365
print("Reproduced. You are ready for notebook 1.")